In [34]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

In [35]:
DATA_PATH = "../data/telco_customer_churn.csv"

df = pd.read_csv(DATA_PATH)



In [36]:
new_customer_missing = (
    df["TotalCharges"].str.strip().eq("")
    & df["tenure"].eq(0)
)

df.loc[
    new_customer_missing,
    "TotalCharges"
] = "0"


df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)


In [37]:
X = df.drop(columns=["customerID", "Churn"])
y = df["Churn"]

In [38]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

In [39]:
overall_rate = y.value_counts(normalize=True).mul(100).round(2)
train_rate = y_train.value_counts(normalize=True).mul(100).round(2)
test_rate = y_test.value_counts(normalize=True).mul(100).round(2)

comparison = pd.DataFrame({
    "Tüm veri (%)": overall_rate,
    "Train (%)": train_rate,
    "Test (%)": test_rate
})

comparison

,Tüm veri (%),Train (%),Test (%)
Churn,,,
No,73.46,73.46,73.46
Yes,26.54,26.54,26.54


In [40]:
numeric_columns = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

categorical_columns = [
    "gender",
    "SeniorCitizen",
    "Partner",
    "Dependents",
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod"
]

In [41]:
numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

In [42]:
categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

Kategorik değişkenlerin kategori sayılarının düşük olması ve hedef
değişkeni kullanmadan uygulanabilmesi nedeniyle one-hot encoding tercih
edilmiştir. Target encoding hedef değişken üzerinden değer ürettiği
için yanlış uygulanması durumunda data leakage oluşturabilir.

In [43]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            numeric_columns
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_columns
        )
    ]
)

In [44]:
preprocessing_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        )
    ]
)

In [45]:
X_train_processed = (
    preprocessing_pipeline.fit_transform(X_train)
)

X_test_processed = (
    preprocessing_pipeline.transform(X_test)
)